In [ ]:
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Resolve absolute paths to local Silver Delta table directories
PROJECT_ROOT = os.path.dirname(os.getcwd())

# Clickstreams
# TARGET_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "silver_clickstream")
# TARGET_DLQ_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "silver_clickstream_dlq")

# Kafka Streams
TARGET_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "silver_kafka_stream")
TARGET_DLQ_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "silver_kafka_stream_dlq")

# Initialize SparkSession with Delta support
builder = (
    SparkSession.builder.appName("Clickstream_Silver_Query")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [7]:
# # CLICKSTREAMS
silver_df = spark.read.format("delta").load(TARGET_PATH)

print(f"Total Clean Silver Records: {silver_df.count()}")
silver_df.select("event_timestamp", "event_date", "event_type", "price_amount", "user_id").show(10, truncate=False)

26/10/17 14:33:26 WARN DeltaLog: Change in the table id detected while updating snapshot. 
Previous snapshot = Snapshot(path=file:/Users/intuitivecode/Code/deltastreams/data/tables/silver_kafka_stream/_delta_log, version=0, metadata=Metadata(16c2b86c-3254-4d37-8125-acecdae94dbe,null,null,Format(parquet,Map()),{"type":"struct","fields":[{"name":"host","type":"string","nullable":true,"metadata":{}},{"name":"port","type":"integer","nullable":true,"metadata":{}},{"name":"source_type","type":"string","nullable":true,"metadata":{}},{"name":"timestamp","type":"string","nullable":true,"metadata":{}},{"name":"src_ip","type":"string","nullable":true,"metadata":{}},{"name":"dst_ip","type":"string","nullable":true,"metadata":{}},{"name":"protocol","type":"string","nullable":true,"metadata":{}},{"name":"src_port","type":"integer","nullable":true,"metadata":{}},{"name":"dst_port","type":"integer","nullable":true,"metadata":{}},{"name":"_processed_at","type":"timestamp","nullable":true,"metadata":{}}

Total Clean Silver Records: 66


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `event_timestamp` cannot be resolved. Did you mean one of the following? [`timestamp`, `dst_ip`, `dst_port`, `_processed_at`, `host`].;
'Project ['event_timestamp, 'event_date, 'event_type, 'price_amount, 'user_id]
+- Relation [host#2428,port#2429,source_type#2430,timestamp#2431,src_ip#2432,dst_ip#2433,protocol#2434,src_port#2435,dst_port#2436,_processed_at#2437] parquet


In [12]:
# KAFKA STREAMS
silver_df = spark.read.format("delta").load(TARGET_PATH)
print(f"Total Clean Network Telemetry Records: {silver_df.count():,}")
silver_df.select("host", "src_ip", "dst_ip", "protocol", "src_port", "dst_port", "_processed_at").show(10, truncate=False)

Total Clean Network Telemetry Records: 311
+----------+-------------+--------+--------+--------+--------+----------------------+
|host      |src_ip       |dst_ip  |protocol|src_port|dst_port|_processed_at         |
+----------+-------------+--------+--------+--------+--------+----------------------+
|172.20.0.1|             |        |        |NULL    |NULL    |2026-10-17 14:34:00.08|
|172.20.0.1|             |        |        |NULL    |NULL    |2026-10-17 14:34:00.08|
|172.20.0.1|192.168.1.100|10.0.0.5|TCP     |443     |5140    |2026-10-17 14:34:00.08|
|172.20.0.1|             |        |        |NULL    |NULL    |2026-10-17 14:34:00.08|
|172.20.0.1|             |        |        |NULL    |NULL    |2026-10-17 14:34:00.08|
|172.20.0.1|             |        |        |NULL    |NULL    |2026-10-17 14:34:00.08|
|172.20.0.1|             |        |        |NULL    |NULL    |2026-10-17 14:34:00.08|
|172.20.0.1|192.168.1.100|10.0.0.5|TCP     |443     |5140    |2026-10-17 14:34:00.08|
|172.20.0.1

In [ ]:
# Load Silver DLQ Delta table
dlq_df = spark.read.format("delta").load(TARGET_DLQ_PATH)

print(f"Total Quarantined DLQ Records: {dlq_df.count()}")
dlq_df.select("event_time", "event_type", "price", "_dlq_reason", "_source_file").show(10, truncate=False)

In [13]:
# KAFKA STREAMS
dlq_df = spark.read.format("delta").load(TARGET_DLQ_PATH)

print(f"Total Quarantined Network Logs (DLQ): {dlq_df.count():,}")
dlq_df.select("host", "message", "_dlq_reason", "_processed_at").show(10, truncate=False)


Total Quarantined Network Logs (DLQ): 156
+----------+------------------------------------------------------------------+-------------------------+----------------------+
|host      |message                                                           |_dlq_reason              |_processed_at         |
+----------+------------------------------------------------------------------+-------------------------+----------------------+
|172.20.0.1|!!! CORRUPT_BYTE_STREAM_INVALID_UTF8_7761527361_NULL_EXCEPTION !!!|MALFORMED_UDP_BYTE_STREAM|2026-10-17 14:34:00.08|
|172.20.0.1|!!! CORRUPT_BYTE_STREAM_INVALID_UTF8_7761527361_NULL_EXCEPTION !!!|MALFORMED_UDP_BYTE_STREAM|2026-10-17 14:34:00.08|
|172.20.0.1|!!! CORRUPT_BYTE_STREAM_INVALID_UTF8_7761527361_NULL_EXCEPTION !!!|MALFORMED_UDP_BYTE_STREAM|2026-10-17 14:34:00.08|
|172.20.0.1|!!! CORRUPT_BYTE_STREAM_INVALID_UTF8_7761527361_NULL_EXCEPTION !!!|MALFORMED_UDP_BYTE_STREAM|2026-10-17 14:34:00.08|
|172.20.0.1|!!! CORRUPT_BYTE_STREAM_INVALID_UTF8_776152